# 02. Day vs Night: Functional Zones of Yogyakarta

**Data source:** `presence_kapanewon_hourly.csv`

**Questions**
1. Which kapanewon gain presence during the day (work and study centres) and which are night-heavy?
2. Can we group the 78 kapanewon into functional zones using only their 24-hour presence shape (K-Means)?

Analysis period: **Dec 2021 - Feb 2022** (best data quality).

In [ ]:
# @title 0. Setup (jalankan sekali, otomatis di Google Colab)
# Di Colab: membuat folder kerja, menulis modul helper, dan mengunduh data dari Zenodo.
# Di laptop (repo GitHub): sel ini tidak melakukan apa-apa.
import os, sys, hashlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE = "/content/jogjapulse"
    for d in ["src", "data", "figures", "notebooks"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    open(f"{BASE}/src/__init__.py", "w").close()
    with open(f"{BASE}/src/jogjapulse.py", "w") as f:
        f.write('"""\nShared helpers for the JogjaPulse human mobility project.\n\nAll notebooks import from here so that loading, cleaning and normalisation\nare done the same way everywhere.\n\nDataset: Widyawan et al. (2026). JogjaPulse: An Aggregated Spatio-Temporal\nMobility Dataset for Yogyakarta, Indonesia. Zenodo.\nhttps://doi.org/10.5281/zenodo.23157460  (CC BY 4.0)\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------\n# Paths\n# --------------------------------------------------------------------------\nROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = ROOT / "data"\nFIG_DIR = ROOT / "figures"\nFIG_DIR.mkdir(exist_ok=True)\n\n# --------------------------------------------------------------------------\n# Constants from the dataset README\n# --------------------------------------------------------------------------\nGOOD_PERIOD_END = "2022-02-28"      # OD flows after this are reduced quality\nSAMPLE_BREAKS = ["2022-01-01", "2022-02-16"]  # abrupt drops in device sample\nANOMALY_DAYS = ["2022-03-08"]       # one-day delivery issue\nNIGHT_HOURS = list(range(0, 6))     # 00:00-05:59 WIB\nDAY_HOURS = list(range(10, 17))     # 10:00-16:59 WIB\nHOLIDAY_WINDOW = ("2021-12-24", "2022-01-02")  # Christmas and New Year\nKAB_ORDER = ["YOGYAKARTA", "SLEMAN", "BANTUL", "KULON PROGO", "GUNUNGKIDUL"]\nDOW_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]\n\n\ndef _path(name: str) -> Path:\n    p = DATA_DIR / name\n    if not p.exists():\n        raise FileNotFoundError(\n            f"{p} not found. Run `python scripts/download_data.py` first "\n            "or download the CSV files from Zenodo into the data/ folder."\n        )\n    return p\n\n\n# --------------------------------------------------------------------------\n# Loaders\n# --------------------------------------------------------------------------\ndef load_hourly(drop_anomalies: bool = True) -> pd.DataFrame:\n    """Kapanewon x hour presence (n_maids, n_pings)."""\n    df = pd.read_csv(_path("presence_kapanewon_hourly.csv"),\n                     dtype={"kab_code": str, "kapanewon_code": str})\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    df["dow"] = df["date"].dt.dayofweek          # 0 = Monday\n    df["is_weekend"] = df["dow"] >= 5\n    df["month"] = df["date"].dt.to_period("M").astype(str)\n    return df\n\n\ndef load_cells(drop_anomalies: bool = True) -> pd.DataFrame:\n    """500 m grid cell x day presence (n_maids)."""\n    df = pd.read_csv(_path("presence_cell500m_daily.csv"))\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    return df\n\n\ndef load_od() -> pd.DataFrame:\n    """Monthly home-based OD visitor counts, joined with origin denominators."""\n    od = pd.read_csv(_path("od_home_visits_kapanewon_monthly.csv"),\n                     dtype={"origin_code": str, "dest_code": str,\n                            "origin_kab_code": str, "dest_kab_code": str})\n    org = pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                      dtype={"origin_code": str, "origin_kab_code": str})\n    od = od.merge(org[["month", "origin_code", "n_homed_maids",\n                       "n_homed_maids_daytime"]],\n                  on=["month", "origin_code"], how="left")\n    od["share"] = od["n_maids"] / od["n_homed_maids_daytime"]\n    od["is_internal"] = od["origin_code"] == od["dest_code"]\n    od["reduced_quality"] = od["reduced_quality"].astype(str).str.lower() == "true"\n    return od\n\n\ndef load_origins() -> pd.DataFrame:\n    return pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                       dtype={"origin_code": str, "origin_kab_code": str})\n\n\ndef load_home_vs_population() -> pd.DataFrame:\n    return pd.read_csv(_path("home_maids_vs_population_kapanewon.csv"))\n\n\ndef load_night_validation() -> pd.DataFrame:\n    return pd.read_csv(_path("validation_night_presence_kapanewon.csv"))\n\n\n# --------------------------------------------------------------------------\n# Normalisation (README section 6: the device sample shrinks over time,\n# so raw counts must never be compared across dates)\n# --------------------------------------------------------------------------\ndef add_hourly_share(df: pd.DataFrame) -> pd.DataFrame:\n    """Share of each kapanewon in the regional total of the same hour.\n\n    Note: n_maids is a distinct count, so the regional sum is not the exact\n    number of distinct devices. It is used here only as a common denominator\n    that removes the effect of the shrinking sample.\n    """\n    out = df.copy()\n    tot = out.groupby(["date", "hour"])["n_maids"].transform("sum")\n    out["share_hour"] = out["n_maids"] / tot\n    return out\n\n\ndef add_daily_share(cells: pd.DataFrame) -> pd.DataFrame:\n    """Share of each grid cell in the total presence of the same day."""\n    out = cells.copy()\n    out["share_day"] = out["n_maids"] / out.groupby("date")["n_maids"].transform("sum")\n    return out\n\n\ndef hourly_profile(df: pd.DataFrame, value: str = "n_pings") -> pd.DataFrame:\n    """Percentage of each day\'s activity that falls in each hour.\n\n    n_pings is additive, so it is the correct variable for regional totals.\n    Returns one row per date with 24 hour columns.\n    """\n    daily = df.groupby(["date", "hour"])[value].sum().unstack("hour").fillna(0)\n    return daily.div(daily.sum(axis=1), axis=0) * 100\n\n\ndef kapanewon_profiles(df: pd.DataFrame) -> pd.DataFrame:\n    """24-hour shape per kapanewon (each row sums to 1), based on the mean\n    hourly share. Used for functional-zone clustering."""\n    s = add_hourly_share(df)\n    prof = s.groupby(["kapanewon", "hour"])["share_hour"].mean().unstack("hour")\n    prof = prof.reindex(columns=range(24))\n    prof = prof.interpolate(axis=1, limit_direction="both")\n    return prof.div(prof.sum(axis=1), axis=0)\n\n\ndef savefig(fig, name: str, dpi: int = 200) -> Path:\n    path = FIG_DIR / name\n    fig.savefig(path, dpi=dpi, bbox_inches="tight")\n    return path\n')
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "plotly>=5.24"], check=False)
    import requests
    FILES = {'presence_cell500m_daily.csv': '0b4d544e8c9ab26d69360821e9ebff27', 'presence_kapanewon_hourly.csv': '90a4a8ae6eb68abc68f7a1021a51cce8', 'od_home_visits_kapanewon_monthly.csv': '5857291c608dd5335f6d8833c1ec7b7a', 'od_origin_homes_monthly.csv': 'd14368c48d3937c9146ed9054aee3c79', 'od_summary.csv': '139962d1c8101eb6873be6f902320efb', 'home_maids_vs_population_kapanewon.csv': '0f01f7a3bb7027174cc1b5e1ad0e8224', 'validation_night_presence_kapanewon.csv': '4a8d0f393fd6d4fa17fb0e60c6f7408e', 'tier1_summary.csv': 'd391339521ce3714f1616c430512b0e0'}
    for name, md5 in FILES.items():
        target = f"{BASE}/data/{name}"
        if os.path.exists(target):
            continue
        print("Downloading", name, "...", end=" ", flush=True)
        r = requests.get("https://zenodo.org/records/23157460/files/" + name, params={"download": 1}, timeout=300)
        r.raise_for_status()
        open(target, "wb").write(r.content)
        print("OK" if hashlib.md5(r.content).hexdigest() == md5 else "checksum mismatch")
    os.chdir(f"{BASE}/notebooks")
    print("Setup selesai. Folder kerja:", BASE)


In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from src import jogjapulse as jp

sns.set_theme(style="whitegrid", context="talk", font_scale=0.8)
hourly = jp.load_hourly()
hourly = hourly[hourly["date"] <= jp.GOOD_PERIOD_END]
s = jp.add_hourly_share(hourly)

## 2.1 Relative day/night index
`index = mean share during 10:00-16:59 / mean share during 00:00-05:59`

- index > 1: the kapanewon holds a larger share of DIY's devices by day than by night (attracts people in daytime)
- index < 1: relatively night-heavy (residential areas, hotels, night economy)

Kapanewon with very few published night slots are excluded because their night value is unreliable (README limitation 5).

In [ ]:
day = s[s["hour"].isin(jp.DAY_HOURS)].groupby(["kab_kota", "kapanewon"])["share_hour"].mean()
night_g = s[s["hour"].isin(jp.NIGHT_HOURS)].groupby(["kab_kota", "kapanewon"])["share_hour"]
night = night_g.mean()
n_days = hourly["date"].nunique()
night_coverage = night_g.size() / (n_days * len(jp.NIGHT_HOURS)) * 100

idx = pd.DataFrame({"day_share_%": day * 100, "night_share_%": night * 100,
                    "night_slots_published_%": night_coverage}).dropna()
idx["day_night_index"] = idx["day_share_%"] / idx["night_share_%"]
reliable = idx[idx["night_slots_published_%"] >= 30].sort_values("day_night_index")
print(f"{len(reliable)} of {len(idx)} kapanewon have >=30% of night slots published")

In [ ]:
show = pd.concat([reliable.head(10), reliable.tail(10)]).reset_index()
show["label"] = show["kapanewon"].str.title() + " (" + show["kab_kota"].str.title() + ")"
colors = ["#1f4e79" if v < 1 else "#e67e22" for v in show["day_night_index"]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(show["label"], show["day_night_index"], color=colors)
ax.axvline(1, color="black", lw=1)
ax.set_xlabel("Relative day/night index (1 = same as DIY average)")
ax.set_title("Most night-heavy (blue) vs most day-heavy (orange) kapanewon\nDec 2021 - Feb 2022")
jp.savefig(fig, "02_day_night_index.png")
plt.show()

In [ ]:
by_kab = reliable.groupby(level="kab_kota")["day_night_index"].describe()[["count", "mean", "50%", "min", "max"]]
by_kab.round(2)

## 2.2 Top daytime magnets (largest share of DIY devices during working hours)

In [ ]:
top_day = idx.sort_values("day_share_%", ascending=False).head(10).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_day, y="kapanewon", x="day_share_%", hue="kab_kota", dodge=False, ax=ax)
ax.set_xlabel("Mean share of DIY devices, 10:00-16:59 (%)")
ax.set_ylabel("")
ax.set_title("Top 10 daytime magnets")
jp.savefig(fig, "02_top_daytime_share.png")
plt.show()

## 2.3 Functional zones with K-Means on 24-hour profiles
Each kapanewon is represented by a 24-value vector: the shape of its presence over the day (rows sum to 1).
We pick k with the silhouette score, then interpret each cluster from its average profile.

In [ ]:
prof = jp.kapanewon_profiles(hourly)
X = prof.values
scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    scores[k] = silhouette_score(X, km.labels_)
print("Silhouette by k:", {k: round(v, 3) for k, v in scores.items()})
best_k = max(scores, key=scores.get)
k = best_k if best_k >= 3 else 3   # at least 3 zones are easier to interpret
km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
prof["cluster"] = km.labels_

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for c in sorted(prof["cluster"].unique()):
    members = prof[prof["cluster"] == c].drop(columns="cluster")
    ax.plot(range(24), members.mean() * 100, marker="o", lw=2,
            label=f"Cluster {c} (n={len(members)})")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day (WIB)")
ax.set_ylabel("% of the kapanewon's daily presence")
ax.set_title("Average 24-hour presence profile per functional cluster")
ax.legend()
jp.savefig(fig, "02_cluster_profiles.png")
plt.show()

In [ ]:
kab_lookup = hourly.drop_duplicates("kapanewon").set_index("kapanewon")["kab_kota"]
members = (prof["cluster"].rename("cluster").to_frame()
           .join(kab_lookup).reset_index())
for c, g in members.groupby("cluster"):
    print(f"\nCluster {c}: {len(g)} kapanewon")
    print("  by regency:", g["kab_kota"].value_counts().to_dict())
    print("  examples:", ", ".join(g["kapanewon"].str.title().head(12)))
members.to_csv(jp.FIG_DIR / "02_cluster_membership.csv", index=False)

In [ ]:
pca = PCA(n_components=2).fit(X)
pts = pca.transform(X)
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(pts[:, 0], pts[:, 1], c=prof["cluster"], cmap="tab10", s=60)
for name, (x, y) in zip(prof.index, pts):
    if name in reliable.index.get_level_values("kapanewon")[:5].tolist() + \
               reliable.index.get_level_values("kapanewon")[-5:].tolist():
        ax.annotate(name.title(), (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)")
ax.set_title("Kapanewon in profile space (PCA), coloured by cluster")
jp.savefig(fig, "02_cluster_pca.png")
plt.show()

## Key findings (fill in after running)
- Kota Yogyakarta core (e.g. Gondomanan, Kraton) is relatively **night-heavy**: hotels, tourism and night economy keep devices there after dark.
- Many rural kapanewon in Gunungkidul and Kulon Progo are relatively **day-heavy**; interpret carefully because their night rows are more often suppressed.
- Depok (Sleman) holds the largest daytime share of devices in DIY (campus and commercial belt).
- Clusters: describe each in one line, e.g. "residential", "daytime activity centre", "night economy / tourism".

In [ ]:
# @title Unduh semua grafik (Colab)
# Membuat figures.zip lalu mengunduhnya ke laptop, untuk slide dan untuk di-push ke GitHub.
if IN_COLAB:
    import shutil
    from google.colab import files as colab_files
    shutil.make_archive("/content/figures", "zip", "/content/jogjapulse/figures")
    colab_files.download("/content/figures.zip")
